[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C67_LLM_Judge_Course/02_bias/02_judge_bias.ipynb)

# 02 · Judge 偏差与去偏（位置 / 长度 / 自偏好 / 风格 · 探针与去偏）

目标：把「我知道 judge 有偏差」升级成「我量过这个 judge 的偏差有多大，并把它从数字里减掉了」。

本 notebook 你会亲手实现：
1. **位置偏差探针与 swap 去偏** —— 区分「位置偏差」与「纯噪声」，两者修法完全不同
2. **长度控制回归** —— 纯 numpy 的逻辑回归，拟合长度系数 γ 并算出 LC 胜率
3. **自偏好探针** —— 在人类判为等质量的样本上，量同族偏好的幅度
4. **风格偏差与去风格化** —— 同内容不同格式的配对检验
5. **多 judge 集成** —— 独立偏差 vs 共有偏差，后者集成完全无用
6. **可识别性演示** —— 长度与质量真相关时，去偏会把真实优势一起减掉

> 心智模型：**去偏的正确输出不是一个更准的数字，而是一个更窄的区间 + 一句关于假设的说明。**

## 1 · 位置偏差探针：区分「偏差」与「噪声」

In [ ]:
import math, json
from collections import Counter, defaultdict
import numpy as np

class BiasedJudge:
    """带四个显式偏差旋钮的 judge。所有偏差都作用在「感知质量」上。"""

    def __init__(self, b_pos=0.0, b_len=0.0, b_self=0.0, b_style=0.0,
                 noise=0.25, seed=0):
        self.b_pos, self.b_len = b_pos, b_len
        self.b_self, self.b_style = b_self, b_style
        self.noise = noise
        self.rng = np.random.default_rng(seed)

    def _v(self, q, is_first, z_len, is_self, is_styled):
        return (q + self.b_pos * is_first + self.b_len * z_len
                + self.b_self * is_self + self.b_style * is_styled
                + self.rng.normal(0, self.noise, size=np.shape(q)))

    def compare(self, qa, qb, a_first=True, z_len_a=0.0, z_len_b=0.0,
                self_a=0, self_b=0, style_a=0, style_b=0):
        """返回 1 = 判 A 赢。a_first 决定谁被放在提示的第一位。"""
        va = self._v(np.asarray(qa, float), 1 if a_first else 0, z_len_a, self_a, style_a)
        vb = self._v(np.asarray(qb, float), 0 if a_first else 1, z_len_b, self_b, style_b)
        return (va > vb).astype(int)


def swap_probe(judge, qa, qb):
    """两种顺序各判一次。返回 (一致率, 位置偏差强度, swap 平均后的 A 胜率)。
    位置偏差强度 = 两种顺序下「选第一个」的平均倾向 - 0.5。"""
    v_ab = judge.compare(qa, qb, a_first=True)      # A 在前，1 表示选了第一个
    v_ba = judge.compare(qa, qb, a_first=False)     # A 在后，1 表示选了第二个
    consistent = float((v_ab == v_ba).mean())
    pick_first = float((v_ab.mean() + (1 - v_ba).mean()) / 2)
    win_a = float((v_ab.mean() + v_ba.mean()) / 2)  # swap 平均：位置项被消掉
    return consistent, pick_first - 0.5, win_a


rng = np.random.default_rng(0)
N = 4000
qa = rng.uniform(0, 1, N)
qb = rng.uniform(0, 1, N)
truth_win = float((qa > qb).mean())

CASES = [
    ('无偏差、低噪声', BiasedJudge(noise=0.10, seed=1)),
    ('纯噪声（无位置偏差）', BiasedJudge(noise=0.45, seed=2)),
    ('纯位置偏差（低噪声）', BiasedJudge(b_pos=0.25, noise=0.10, seed=3)),
    ('位置偏差 + 噪声', BiasedJudge(b_pos=0.25, noise=0.45, seed=4)),
]
print(f"{'judge':<24}{'swap 一致率':>12}{'位置偏差':>10}{'swap 平均胜率':>14}")
for name, j in CASES:
    c, bias, wa = swap_probe(j, qa, qb)
    print(f'{name:<24}{c:>12.1%}{bias:>+10.3f}{wa:>14.1%}')
print(f'{"真值":<24}{"":>12}{0.0:>+10.3f}{truth_win:>14.1%}')

c_noise, bias_noise, _ = swap_probe(BiasedJudge(noise=0.45, seed=2), qa, qb)
c_pos, bias_pos, _ = swap_probe(BiasedJudge(b_pos=0.25, noise=0.10, seed=3), qa, qb)
assert abs(bias_noise) < 0.05, '纯噪声不应产生位置偏差'
assert bias_pos > 0.05, '纯位置偏差必须被探针检出'
assert c_noise < 0.95 and c_pos < 0.95, '两者都会降低 swap 一致率'
print('\n✅ 关键结论：**两个 judge 的 swap 一致率都不高，但成因完全不同**——')
print('   噪声型：一致率低但位置偏差 ≈ 0 → 修法是多次采样或换更强的 judge')
print('   偏差型：一致率低且位置偏差显著 → 修法是 swap 平均（本函数已经做了）')

In [ ]:
# swap 平均确实把位置偏差消掉了：与只做单向调用对比
j = BiasedJudge(b_pos=0.30, noise=0.15, seed=7)
one_way = float(j.compare(qa, qb, a_first=True).mean())
_, _, swapped = swap_probe(BiasedJudge(b_pos=0.30, noise=0.15, seed=7), qa, qb)
print(f'真值 A 胜率        {truth_win:.1%}')
print(f'单向调用（A 在前） {one_way:.1%}   偏离 {one_way - truth_win:+.1%}')
print(f'swap 平均          {swapped:.1%}   偏离 {swapped - truth_win:+.1%}')
assert abs(swapped - truth_win) < abs(one_way - truth_win)
print('\n✅ swap 平均把位置偏差从数字里消掉了——代价是调用次数翻倍。')
print('   这是四个偏差里唯一有干净修法的一个，因为「交换顺序」是严格保内容的操作。')

## 2 · 长度控制回归：拟合 γ，算出 LC 胜率

$$\operatorname{logit}\Pr(A \succ B) = \theta + \gamma\cdot\Delta_{\text{len}}$$

拟合完把 $\Delta_{\text{len}}$ 设为 0，就得到长度控制后的胜率。纯 numpy 的梯度下降。

In [ ]:
def fit_logistic(X, y, lr=0.5, iters=3000, l2=1e-4):
    """纯 numpy 逻辑回归（含截距）。X: (n, d)，y: (n,) 取值 0/1。返回系数向量（末位是截距）。"""
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    Xb = np.hstack([X, np.ones((len(X), 1))])
    w = np.zeros(Xb.shape[1])
    for _ in range(iters):
        z = Xb @ w
        p = 1 / (1 + np.exp(-np.clip(z, -30, 30)))
        grad = Xb.T @ (p - y) / len(y) + l2 * w
        w -= lr * grad
    return w

def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -30, 30)))

# 造数据：A 比 B 真实质量高 delta_q，同时 A 平均更长
rng = np.random.default_rng(11)
M = 6000
delta_q = 0.06
base = rng.uniform(0, 0.9, M)
qA, qB = base + delta_q, base
len_A = rng.normal(600, 200, M)
len_B = rng.normal(380, 160, M)
dlen = (len_A - len_B) / (len_A + len_B)             # 归一化长度差（关键：不要用原始差值）

judge_len = BiasedJudge(b_len=0.0, noise=0.30, seed=13)
# 长度偏差直接体现在感知质量上：A 每单位归一化长度差 +0.9
va = qA + 0.9 * dlen + rng.normal(0, 0.30, M)
vb = qB + rng.normal(0, 0.30, M)
y = (va > vb).astype(int)

raw_win = float(y.mean())
w = fit_logistic(dlen.reshape(-1, 1), y)
gamma, theta = float(w[0]), float(w[1])
lc_win = float(sigmoid(theta))                       # 把 dlen 置 0
print(f'原始胜率 (raw)       {raw_win:.1%}')
print(f'长度系数 gamma       {gamma:+.3f}')
print(f'长度控制后胜率 (LC)  {lc_win:.1%}')
assert gamma > 0.5, '长度偏差必须被拟合出来'
assert lc_win < raw_win, '控制掉长度优势后，胜率应当下降'
print('\n✅ 8 个点里有一大半来自「A 写得更长」，而不是「A 写得更好」。')

In [ ]:
# 好的去偏方法应有的性质：没有偏差时不引入伤害
rng = np.random.default_rng(21)
va2 = qA + 0.0 * dlen + rng.normal(0, 0.30, M)       # 这次 judge 没有长度偏差
vb2 = qB + rng.normal(0, 0.30, M)
y2 = (va2 > vb2).astype(int)
w2 = fit_logistic(dlen.reshape(-1, 1), y2)
raw2, lc2 = float(y2.mean()), float(sigmoid(w2[1]))
print(f'无长度偏差的 judge: gamma = {w2[0]:+.3f} | raw {raw2:.1%} → LC {lc2:.1%}（几乎不变）')
assert abs(float(w2[0])) < 0.25
assert abs(lc2 - raw2) < 0.03
print('\n✅ γ 被拟合成接近 0，去偏后的胜率与去偏前几乎一样——')
print('   **方法自动退化成「什么都不做」，这是一个好的去偏方法应有的性质。**')
print('   对比「直接惩罚长回答」那种做法：它在没有偏差时会主动制造偏差。')

In [ ]:
# 为什么必须归一化长度差：用原始 token 差会让长回答占据不成比例的杠杆
raw_dlen = (len_A - len_B)
w_raw = fit_logistic(raw_dlen.reshape(-1, 1), y)
lc_raw_scale = float(sigmoid(w_raw[1]))
print(f'归一化长度差:  gamma={gamma:+.4f}      LC 胜率 {lc_win:.1%}')
print(f'原始 token 差: gamma={float(w_raw[0]):+.6f}  LC 胜率 {lc_raw_scale:.1%}')
print(f'\n两种做法的 LC 胜率差 {abs(lc_win - lc_raw_scale):.1%}')
print('✅ 归一化不是洁癖：原始差值的量纲让极端长的样本获得极大杠杆，')
print('   系数估计对少数长样本非常敏感。推荐用 (a-b)/(a+b) 或 log 长度之差。')

## 3 · 自偏好探针：在「人类判为等质量」的样本上量

In [ ]:
# 设计：取一批真实质量严格相等的成对样本（构造性地保证），
# 其中一个由 judge 的同族模型生成。任何系统性偏向都是自偏好。
rng = np.random.default_rng(31)
K = 3000
q_equal = rng.uniform(0.2, 0.8, K)                  # 两边质量完全相同
is_self_a = np.ones(K)                              # A 由同族模型生成
is_self_b = np.zeros(K)

def self_pref_probe(b_self, noise=0.25, seed=0):
    j = BiasedJudge(b_self=b_self, noise=noise, seed=seed)
    v_ab = j.compare(q_equal, q_equal, a_first=True, self_a=is_self_a, self_b=is_self_b)
    v_ba = j.compare(q_equal, q_equal, a_first=False, self_a=is_self_a, self_b=is_self_b)
    return float((v_ab.mean() + v_ba.mean()) / 2)   # swap 平均，排除位置偏差干扰

print(f"{'自偏好强度 b_self':>18}{'同族模型的胜率':>16}{'偏离 50%':>12}")
for b in [0.0, 0.05, 0.10, 0.20]:
    wr = self_pref_probe(b, seed=int(b * 100) + 1)
    print(f'{b:>18.2f}{wr:>16.1%}{wr-0.5:>+12.1%}')

wr0 = self_pref_probe(0.0, seed=1)
wr2 = self_pref_probe(0.20, seed=21)
assert abs(wr0 - 0.5) < 0.03, '无自偏好时应当是 50/50'
assert wr2 > 0.55, '自偏好必须被检出'
print('\n✅ 探针的关键在于「质量构造性地相等」——')
print('   在真实数据上做这个探针，必须用人类判为平局的子集，否则无法区分')
print('   「偏爱自己」与「自己确实更好」这两种解释。')

In [ ]:
# 最危险的配置：用模型 M 做 judge 来评测 M 的新旧版本
rng = np.random.default_rng(37)
n = 4000
q_old = rng.uniform(0, 1, n)
true_gain = 0.02                                     # 新版本真实只强 2 个点
q_new = q_old + true_gain

def measured_gain(b_self_new, seed):
    j = BiasedJudge(b_self=b_self_new, noise=0.25, seed=seed)
    # 新版本的输出更像 judge 自己（judge 就是新版本的同族）
    v1 = j.compare(q_new, q_old, a_first=True, self_a=np.ones(n), self_b=np.zeros(n))
    v2 = j.compare(q_new, q_old, a_first=False, self_a=np.ones(n), self_b=np.zeros(n))
    return float((v1.mean() + v2.mean()) / 2)

wr_no_self = measured_gain(0.0, 41)
wr_self = measured_gain(0.12, 42)
print(f'真实质量提升             {true_gain:+.1%}')
print(f'异族 judge 测出的胜率     {wr_no_self:.1%}')
print(f'同族 judge 测出的胜率     {wr_self:.1%}  ← 虚高 {wr_self - wr_no_self:+.1%}')
assert wr_self > wr_no_self + 0.03
print('\n✅ 这正是最常见的内部评测配置——用自家模型评自家新版本。')
print('   缓解的最低限度：用一个异族 judge 交叉验证，至少确认排序一致。')

## 4 · 风格偏差：同内容、不同格式的配对检验

In [ ]:
def paired_t(diff):
    """配对样本的 t 统计量与近似双侧 p 值（正态近似）。"""
    d = np.asarray(diff, dtype=float)
    t = d.mean() / (d.std(ddof=1) / math.sqrt(len(d)) + 1e-12)
    p = math.erfc(abs(t) / math.sqrt(2))
    return float(t), float(p)

rng = np.random.default_rng(51)
n_pairs = 400
q_same = rng.uniform(0.2, 0.8, n_pairs)             # 同一份内容，质量当然相同

def style_probe(b_style, seed):
    j = BiasedJudge(b_style=b_style, noise=0.25, seed=seed)
    # 左边是「markdown 重排版」，右边是「纯文本版」，内容完全相同
    v1 = j.compare(q_same, q_same, a_first=True, style_a=1, style_b=0)
    v2 = j.compare(q_same, q_same, a_first=False, style_a=1, style_b=0)
    wins = (v1 + v2) / 2                             # 每对样本的「结构化版胜出」得分
    return float(wins.mean()), paired_t(wins - 0.5)

for b in [0.0, 0.10, 0.25]:
    wr, (t, p) = style_probe(b, seed=int(b * 100) + 61)
    print(f'风格偏差 {b:.2f} → 结构化版胜率 {wr:.1%} | 配对 t={t:+.2f} p={p:.2e}')

wr0, (t0, p0) = style_probe(0.0, 61)
wr2, (t2, p2) = style_probe(0.25, 86)
assert p0 > 0.01, '无风格偏差时不应显著'
assert p2 < 0.001 and wr2 > 0.6, '有风格偏差时必须被检出'
print('\n✅ 400 对样本就足以把风格偏差检出到 p < 0.001——')
print('   配对检验消掉了「内容质量」这个最大的方差源（呼应 C66-04 的配对设计）。')
print('   注意：格式算不算质量，取决于你的构念定义。rubric 化能把它变成显式加权的一项。')

## 5 · 多 judge 集成：独立偏差有用，共有偏差完全无用

In [ ]:
def ensemble_experiment(bias_mode, K_judges=5, n=4000, seed=0):
    """bias_mode: 'noise'(纯噪声) | 'independent'(方向各异的偏差) | 'shared'(共有偏差)"""
    rng = np.random.default_rng(seed)
    q1 = rng.uniform(0, 1, n)
    q2 = rng.uniform(0, 1, n)
    truth = (q1 > q2).astype(int)
    votes = np.zeros(n)
    for k in range(K_judges):
        if bias_mode == 'noise':
            b = 0.0
        elif bias_mode == 'independent':
            b = 0.30 * (1 if k % 2 == 0 else -1)     # 方向交替
        else:
            b = 0.30                                  # 所有 judge 同方向
        j = BiasedJudge(b_pos=b, noise=0.35, seed=seed * 100 + k)
        votes += j.compare(q1, q2, a_first=True)
    maj = (votes > K_judges / 2).astype(int)
    single = BiasedJudge(b_pos=(0.0 if bias_mode == 'noise' else 0.30),
                         noise=0.35, seed=seed * 100 + 999).compare(q1, q2, a_first=True)
    return float((single == truth).mean()), float((maj == truth).mean())

print(f"{'误差类型':<22}{'单 judge':>12}{'5-judge 投票':>14}{'提升':>10}")
for mode, label in [('noise', '纯随机噪声'), ('independent', '方向各异的偏差'),
                    ('shared', '共有的同方向偏差')]:
    s, m = ensemble_experiment(mode, seed=7)
    print(f'{label:<22}{s:>12.1%}{m:>14.1%}{m-s:>+10.1%}')

s_n, m_n = ensemble_experiment('noise', seed=7)
s_s, m_s = ensemble_experiment('shared', seed=7)
s_i, m_i = ensemble_experiment('independent', seed=7)
assert (m_n - s_n) > (m_s - s_s), '集成对噪声的收益必须大于对共有偏差的收益'
assert (m_i - s_i) > (m_s - s_s), '方向各异的偏差也能被集成部分抵消'
print('\n✅ 共有偏差那一行的提升最小——集成只能治噪声，不能治所有 judge 共有的偏差。')
print('   而长度偏差、格式偏差恰恰在几乎所有主流 judge 上方向一致。')
print('   **更糟的是：集成会让置信区间变窄，让你更自信地相信一个偏了的结论。**')

## 6 · 可识别性：长度与质量真相关时，去偏会矫枉过正

In [ ]:
def lc_with_true_correlation(frac_quality, n=6000, seed=0):
    """A 比 B 长的部分中，有 frac_quality 的比例是「因为更完整而更长」（真实质量），
    其余是纯注水。看长度控制回归会把多少真实优势一起减掉。"""
    rng = np.random.default_rng(seed)
    base = rng.uniform(0, 0.85, n)
    extra_len = np.abs(rng.normal(0.25, 0.12, n))          # A 多出来的归一化长度
    q_gain = frac_quality * 0.4 * extra_len                # 其中真实提升质量的部分
    qA_, qB_ = base + q_gain, base
    va_ = qA_ + 0.9 * extra_len + rng.normal(0, 0.3, n)    # judge 还有 0.9 的长度偏差
    vb_ = qB_ + rng.normal(0, 0.3, n)
    yy = (va_ > vb_).astype(int)
    ww = fit_logistic(extra_len.reshape(-1, 1), yy)
    raw = float(yy.mean())
    lc = float(sigmoid(ww[1]))
    # 真值：没有长度偏差时的胜率
    va_true = qA_ + rng.normal(0, 0.3, n)
    true_wr = float((va_true > (qB_ + rng.normal(0, 0.3, n))).mean())
    return raw, lc, true_wr

print(f"{'长度中真实质量占比':>20}{'raw':>10}{'LC':>10}{'真值':>10}{'LC 的误差':>12}")
for f in [0.0, 0.3, 0.6, 1.0]:
    r_, l_, t_ = lc_with_true_correlation(f, seed=71)
    print(f'{f:>20.0%}{r_:>10.1%}{l_:>10.1%}{t_:>10.1%}{l_-t_:>+12.1%}')

r0, l0, t0 = lc_with_true_correlation(0.0, seed=71)
r1, l1, t1 = lc_with_true_correlation(1.0, seed=71)
assert abs(l0 - t0) < abs(l1 - t1), '长度全是注水时 LC 最准；长度全是质量时 LC 矫枉过正'
assert l1 < t1, 'LC 把真实优势也减掉了'
print('\n✅ 第一行（长度全是注水）：LC 几乎等于真值——去偏做对了。')
print('   最后一行（长度全来自更完整）：LC 显著低于真值——**把真实优势一起减掉了**。')
print('   而观测数据无法区分这两种情形（不可识别）。')
print('   → 诚实的报告方式：**给区间**「raw 62%，LC 54%，真值在两者之间」。')

## ✏️ 练习 1：位置偏差与噪声的分离

先记一个恒等式（值得单独理解）：设不一致率 $u = 1 - \text{一致率}$。

- **一致**的样本对，两种顺序选的是同一个答案 → 一次选到第一位、一次选到第二位 → 平均贡献 0.5；
- 因**位置偏差**而不一致的样本对，两次都选了「排在前面的那个」→ 贡献 1.0；
- 因**噪声**而不一致的样本对，翻转方向是随机的 → 平均仍贡献 0.5。

所以 $\text{位置偏差强度} = \Pr(\text{选第一个}) - 0.5 = \tfrac{1}{2}u_{\text{position}}$，
即 **$u$ 中由位置偏差解释的比例 = $2\times\text{位置偏差} / u$**。

实现 `diagnose_swap(v_ab, v_ba)`：输入两种顺序下的判断
（`v_ab[i]=1` 表示顺序 AB 时判 A 赢；`v_ba[i]=1` 表示顺序 BA 时判 A 赢），
返回 `(一致率, 位置偏差强度, 解释比例, 诊断)`，诊断规则：
`u < 0.08` → `'clean'`；否则 `ratio ≥ 0.6` → `'position'`，
`ratio ≤ 0.25` → `'noise'`，其余 → `'both'`。

In [ ]:
def diagnose_swap(v_ab, v_ba):
    # TODO：位置偏差强度 = ((v_ab 选第一个的比例) + (v_ba 选第一个的比例)) / 2 - 0.5
    # 注意 v_ba 里「选第一个」等价于「没选 A」；解释比例 = 2*偏差/不一致率
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
def gen(judge):
    return judge.compare(qa, qb, a_first=True), judge.compare(qa, qb, a_first=False)

for name, j, expect in [
        ('无偏差低噪声', BiasedJudge(noise=0.03, seed=101), 'clean'),
        ('纯噪声',       BiasedJudge(noise=0.60, seed=102), 'noise'),
        ('纯位置偏差',   BiasedJudge(b_pos=0.30, noise=0.02, seed=103), 'position'),
        ('两者兼有',     BiasedJudge(b_pos=0.20, noise=0.60, seed=104), 'both')]:
    c, b, r, d = diagnose_swap(*gen(j))
    print(f'{name:<14} 一致率 {c:.1%} | 位置偏差 {b:+.3f} | 位置解释了 {r:5.0%} 的不一致 | 诊断 {d}')
    assert d == expect, f'{name} 期望 {expect} 实际 {d}'
print('\n✅ 练习 1 通过：注意「纯噪声」和「纯位置偏差」两行的一致率都在五成左右——')
print('   只看一致率完全分不开它们，而「位置解释了多少不一致」这个比例一眼就分开了。')
print('   噪声型 → 多采样或换更强的 judge；偏差型 → swap 平均。')

## ✏️ 练习 2：长度控制后的胜率与区间

实现 `lc_report(dlen, y)`：返回字典
`{'raw': 原始胜率, 'gamma': 长度系数, 'lc': 长度控制后胜率, 'interval': (min, max)}`，
其中 `interval` 是 `(min(raw, lc), max(raw, lc))`——即第 6 节主张的「诚实区间」。

In [ ]:
def lc_report(dlen, y):
    # TODO：用上面的 fit_logistic 与 sigmoid
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
rep = lc_report(dlen, y)
assert set(rep) == {'raw', 'gamma', 'lc', 'interval'}
assert rep['interval'][0] <= rep['raw'] <= rep['interval'][1]
assert rep['interval'][0] <= rep['lc'] <= rep['interval'][1]
assert rep['gamma'] > 0.5
rep2 = lc_report(dlen, y2)                       # 无长度偏差的那批数据
assert abs(rep2['interval'][1] - rep2['interval'][0]) < abs(rep['interval'][1] - rep['interval'][0])
for k_, v_ in rep.items():
    print(f'  {k_:<10} {v_}')
print(f"\n无长度偏差时区间宽度 {rep2['interval'][1]-rep2['interval'][0]:.3f}"
      f" < 有偏差时 {rep['interval'][1]-rep['interval'][0]:.3f}")
print('✅ 练习 2 通过：区间宽度本身就是「长度偏差有多大」的直接读数。')

## ✏️ 练习 3：集成的边际收益

实现 `ensemble_gain(mode, K_list, seed=0)`：对每个 K 返回 `(K, 多数投票准确率)`。
用它验证「共有偏差下，增加 judge 数量几乎不提升准确率」。

In [ ]:
def ensemble_gain(mode, K_list, seed=0):
    # TODO：复用 ensemble_experiment，返回 [(K, maj_acc), ...]
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
KS = [1, 3, 5, 9]
g_noise = ensemble_gain('noise', KS, seed=5)
g_shared = ensemble_gain('shared', KS, seed=5)
print(f"{'K':>4}{'纯噪声':>12}{'共有偏差':>12}")
for (k1, a1), (k2, a2) in zip(g_noise, g_shared):
    print(f'{k1:>4}{a1:>12.1%}{a2:>12.1%}')
gain_noise = g_noise[-1][1] - g_noise[0][1]
gain_shared = g_shared[-1][1] - g_shared[0][1]
assert gain_noise > gain_shared
print(f'\nK 从 1 加到 9：纯噪声提升 {gain_noise:+.1%} | 共有偏差提升 {gain_shared:+.1%}')
print('✅ 练习 3 通过：共有偏差下，加再多 judge 也换不来准确率——')
print('   只会换来更窄的置信区间，也就是「更自信地相信一个偏了的结论」。')

## ✏️ 练习 4：探针范式的通用实现

实现 `counterfactual_probe(judge_fn, q, attr_a, attr_b, n_repeat=2)`：
`judge_fn(q, q, attr_a, attr_b, a_first)` 返回 0/1（A 是否胜出）。
在 `a_first=True/False` 各跑一次取平均（消位置偏差），
返回 `(带属性一方的胜率, 配对 t, 配对 p)`。

In [ ]:
def counterfactual_probe(judge_fn, q, attr_a, attr_b):
    # TODO：调用两次（a_first True/False），取每对的平均得分，再做配对检验（用 paired_t）
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
def make_style_fn(b_style, seed):
    j = BiasedJudge(b_style=b_style, noise=0.25, seed=seed)
    def fn(qa_, qb_, sa, sb, a_first):
        return j.compare(qa_, qb_, a_first=a_first, style_a=sa, style_b=sb)
    return fn

wr_a, t_a, p_a = counterfactual_probe(make_style_fn(0.0, 201), q_same,
                                      np.ones(n_pairs), np.zeros(n_pairs))
wr_b, t_b, p_b = counterfactual_probe(make_style_fn(0.25, 202), q_same,
                                      np.ones(n_pairs), np.zeros(n_pairs))
print(f'无风格偏差: 胜率 {wr_a:.1%}  t={t_a:+.2f}  p={p_a:.3f}')
print(f'有风格偏差: 胜率 {wr_b:.1%}  t={t_b:+.2f}  p={p_b:.2e}')
assert p_a > 0.01 and p_b < 0.001
assert wr_b > wr_a
print('✅ 练习 4 通过：同一个函数可以测任何你怀疑的表面属性——')
print('   位置、长度、格式、署名、虚构引用、自信语气。这就是探针范式的价值。')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def diagnose_swap(v_ab, v_ba):
    v_ab = np.asarray(v_ab, dtype=float)
    v_ba = np.asarray(v_ba, dtype=float)
    consistent = float((v_ab == v_ba).mean())
    u = 1 - consistent
    pick_first = float((v_ab.mean() + (1 - v_ba).mean()) / 2)
    bias = pick_first - 0.5
    ratio = (2 * bias / u) if u > 1e-12 else 0.0
    if u < 0.08:
        d = 'clean'
    elif ratio >= 0.6:
        d = 'position'
    elif ratio <= 0.25:
        d = 'noise'
    else:
        d = 'both'
    return (consistent, bias, ratio, d)

In [ ]:
# 练习 2 参考答案
def lc_report(dlen, y):
    w = fit_logistic(np.asarray(dlen).reshape(-1, 1), y)
    raw = float(np.asarray(y, dtype=float).mean())
    lc = float(sigmoid(w[1]))
    return {'raw': round(raw, 4), 'gamma': round(float(w[0]), 4),
            'lc': round(lc, 4), 'interval': (round(min(raw, lc), 4), round(max(raw, lc), 4))}

In [ ]:
# 练习 3 参考答案
def ensemble_gain(mode, K_list, seed=0):
    out = []
    for K in K_list:
        _, maj = ensemble_experiment(mode, K_judges=K, seed=seed)
        out.append((K, maj))
    return out

In [ ]:
# 练习 4 参考答案
def counterfactual_probe(judge_fn, q, attr_a, attr_b):
    v1 = judge_fn(q, q, attr_a, attr_b, True)
    v2 = judge_fn(q, q, attr_a, attr_b, False)
    scores = (np.asarray(v1, dtype=float) + np.asarray(v2, dtype=float)) / 2
    t, p = paired_t(scores - 0.5)
    return (float(scores.mean()), t, p)

---
## 🧪 真实工程胶囊：偏差探针的落地脚本

In [ ]:
RECIPE = r'''
# ══════════════════════════════════════════════════════════════════
# A. swap 探针（每次评测都必须跑，不是可选项）
# ══════════════════════════════════════════════════════════════════
async def judge_with_swap(client, item):
    v1 = await call_judge(client, item.request, item.a, item.b)          # 顺序 AB
    v2 = await call_judge(client, item.request, item.b, item.a)          # 顺序 BA
    v2 = {"A": "B", "B": "A", "tie": "tie"}[v2]        # ← 必须翻译回来，这个 bug 极常见
    return {"v_ab": v1, "v_ba": v2, "consistent": v1 == v2,
            "final": v1 if v1 == v2 else "tie"}
# 报告：swap 一致率 + 位置偏差强度。一致率 < 0.80 时，胜率数字不要用。

# ══════════════════════════════════════════════════════════════════
# B. 长度控制回归（AlpacaEval-LC 的简化实现思路）
# ══════════════════════════════════════════════════════════════════
import numpy as np
def lc_win_rate(df):
    # df 需要列: y (A是否胜), len_a, len_b。返回 raw / gamma / lc
    dlen = (df.len_a - df.len_b) / (df.len_a + df.len_b)     # 归一化，别用原始差
    w = fit_logistic(dlen.values.reshape(-1, 1), df.y.values)
    return {"raw": df.y.mean(), "gamma": w[0], "lc": sigmoid(w[1])}
# 注意：gamma 要**按 judge 分别估计**。不同 judge 的长度偏差差别很大。

# ══════════════════════════════════════════════════════════════════
# C. 风格探针：造对照样本的最省事做法
# ══════════════════════════════════════════════════════════════════
REFORMAT_PROMPT = "
".join([
    "Rewrite the text below into markdown with a heading and",
    "a numbered list. Do NOT add, remove, or change any information.",
    "Return only the rewritten text.",
    "",
    "<text>{text}</text>",
])
# 用一个**独立于被测 judge** 的模型做重排，避免把 judge 自己的风格偏好引进来。
# 重排后人工抽查 20 条，确认信息量确实没变——这一步不能省。

# ══════════════════════════════════════════════════════════════════
# D. 每次评测都该产出的偏差报告块（贴进 eval card）
# ══════════════════════════════════════════════════════════════════
BIAS_BLOCK = "
".join([
    "judge:            {model}, prompt {prompt_hash}, temp {temp}",
    "swap consistency: {swap_consistency:.3f}   (< 0.80 时胜率不可用)",
    "position bias:    {position_bias:+.3f}",
    "win rate (raw):   {raw:.1%}  [{raw_lo:.1%}, {raw_hi:.1%}]",
    "win rate (LC):    {lc:.1%}   gamma={gamma:+.3f}",
    "  └─ 假设: 长度差全部归因于注水；真值在 raw 与 LC 之间",
    "style probe:      {style_delta:+.3f} (n={style_n}, paired p={style_p:.1e})",
    "self-pref probe:  {self_delta:+.3f} (n={self_n})",
])
'''
print(RECIPE)

### 小结

| 你学到的 | 一句话 | 用在哪 |
|---|---|---|
| 四大偏差 | 位置 / 长度 / 自偏好 / 风格，各有探针与去偏手段 | judge 体检 |
| swap 诊断 | 一致率低有两种成因，位置偏差与噪声修法完全不同 | 每次评测 |
| 长度控制回归 | 估计 γ 而不是假设它；没有偏差时自动退化成不做事 | 报告 LC 胜率 |
| 自偏好 | 最危险的配置是「用 M 评 M 的新版本」 | 内部评测 |
| 探针范式 | 造只差一个属性的成对样本 + 配对检验，几十到几百对就够 | 测任何新怀疑的偏差 |
| 集成的边界 | 只能治噪声，治不了所有 judge 共有的偏差 | 决定要不要多 judge |
| 不可识别 | 长度里有多少是真实质量无法从数据分离 | 诚实报区间 |

下一模块：**03 · 元评测与校准**——judge 到底有多准？人类自身的一致率上界是多少？
以及「一致率 82%」这个数字到底该怎么读。